# LandRisk AI — Preprocessing and Model Development

## 1. Project Overview

LandRisk AI is a personal portfolio and learning project inspired by SIH 2026 problem statement **SIH26017 — Predictive Analytics System for Early Detection of Land Acquisition Delays**.

The objective is to develop a machine-learning decision-support prototype that identifies infrastructure projects that may be at risk of significant delay using project-status information.

This project demonstrates an end-to-end machine-learning workflow, including data preparation, feature engineering, model comparison, evaluation, and model selection.

**Dataset:** Synthetic infrastructure project data created for educational purposes.

**Target variable:** `delay_risk`
- `0` — Lower delay risk
- `1` — Higher delay risk

**Important limitation:** Results obtained from synthetic data do not establish real-world predictive accuracy for infrastructure projects.

## 2. Modeling Workflow

1. Load and inspect the dataset.
2. Define the input features and target.
3. Split the data into training and test sets.
4. Build leakage-safe preprocessing and feature-engineering pipelines.
5. Establish a baseline and compare classification models.
6. Evaluate models using appropriate classification metrics.
7. Tune candidate models and select a decision threshold.
8. Evaluate the selected model on the untouched test set.
9. Prepare the complete pipeline for later integration with Django.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
# Data Preprocessing library..
import sklearn
from sklearn.base import clone
sklearn.set_config(display="diagram")
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer, make_column_selector

In [ ]:
# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [ ]:
# Ensemble Models
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

In [ ]:
# Model Evaluation
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix , precision_score, recall_score, accuracy_score
from sklearn.metrics import f1_score, classification_report
from sklearn.metrics import precision_recall_curve , roc_curve, roc_auc_score

In [ ]:
import sys
ROOT = Path.cwd().resolve().parents[1]   # this notebook is in ml/notebooks, so two levels up is the repo root
sys.path.insert(0, str(ROOT))

from ml.features import LandRiskFeatureEngineer

What exactly are we giving to the model, and what exactly are we asking it to predict?

That X and Y
- X is Features 
- Y is Target

Y - Our Target is delay_risk...Cause our main purpose was going to figure out that the project that the current snapshot was take is that project is going to be Delay (Show the Sign of delay)...or the Project is going to be ready at planned_duration_months.

- delay_risk :- 0 → lower delay risk, 1 → higher delay risk.

X - Features
Everything the model is allowed to know before making the prediction.

1. sector
2. state
3. project_cost_cr
4. planned_duration_months
5. months_elapsed
6. physical_progress_pct
7. expenditure_pct
8. contractor_experience_years
9. land_acquisition_required
10. land_acquisition_pct

land_acquisition_required is categorical, not numerical
Even though it is stored as 0 and 1, those numbers are labels, not measurements.
For our dataset:
0 → Land acquisition is NOT required
1 → Land acquisition IS required

In [ ]:
# Loading Dataset
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()

DATA_PATH = NOTEBOOK_DIR.parent/"data"/"infrastructure_projects.csv"
data = pd.read_csv(DATA_PATH)
print(f"Dataset Shape:- {data.shape}")
data.head(3)

In [ ]:
data.info()

**Numerical features** - 

1. project_cost_cr
2. planned_duration_months
3. months_elapsed
4. physical_progress_pct
5. expenditure_pct
6. contractor_experience_years
7. land_acquisition_pct

**Categorical features** - 

1. sector
2. state
3. land_acquisition_required

But here the feature like project_id, project_name are going to be drop cause that hold no value for our model learning and either there is no Relationship about project name , id with the is project going to delay or not..

and Target -> delay_risk

In [ ]:
x = data.drop(columns=['project_id', 'project_name','delay_risk'] , axis= 1)
x.head(2)

In [ ]:
y = data['delay_risk']
y.head(2)

In [ ]:
# Spliting the Data (x and y) into train, test
X_train, X_test , y_train, y_test = train_test_split(x,y,random_state=42,test_size=0.2,stratify=y)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

In [ ]:
print("Total Null Values in the Training Set: ", X_train.isna().sum().sum())
print("Total Null Values in the Test Set: ", X_test.isna().sum().sum())

Why creating Test set?
- if you look at the test set, you may stumble upon some seemingly interesting pattern in the test data that leads you to select a particular kind of machine learning
model. 
- When you estimate the generalization error using the test set, your estimate will be too optimistic, and you will launch a system that will not perform as well as expected
- Test Set help to get and idea about how actual our model has been train perfect and how it will work on the real world data...

**Prepare the Data for Machine Learning Algorithms**

In [ ]:
X_train.head(3)

In [ ]:
y_train.head(3)

**Clean the Data**

1) Most machine learning algorithms cannot work with missing features, so you’ll need to take care of these.
For Example, in our data expenditure_pct, contractor_experience_years holds some missing values 
### 3.1 Missing-Value Analysis

Missing values can prevent machine-learning algorithms from fitting successfully. Rather than deleting projects unnecessarily, this workflow imputes missing numerical values using the median and missing categorical values using the most frequent category.

Imputation statistics are learned from the training data within the preprocessing pipeline, preventing information from the validation or test sets from influencing them.

You have three options to fix this:
1. Get rid of the corresponding districts (rows).
2. Get rid of the whole attribute (Feature). 
3. Set the missing values to some value (zero, the mean, the median, etc.). This is called imputation.

In [ ]:
missing_summary = (
    X_train.isna()
    .sum()
    .rename("Missing Values")
    .to_frame()
)

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
]

display(missing_summary)

In [ ]:
X_train.isna().sum().sum()  # 53 Null Values in the trainset

In [ ]:
x_train_copy = X_train.copy()
x_test_copy = X_test.copy()

In [ ]:
NUMERICAL_ATTRIBUTES = x_train_copy.select_dtypes(include=[np.number])
CATEGORICAL_ATTRIBUTES = x_train_copy.select_dtypes(include=[np.object_])

In [ ]:
imputer = SimpleImputer(strategy="median")
imputer.fit(NUMERICAL_ATTRIBUTES)                  # The imputer has simply computed the median of each attribute and stored the result in its statistics_ instance variable

There are also more powerful imputers available in the sklearn.impute package (both for numerical features only):
1) KNNImputer replaces each missing value with the mean of the k-nearest neighbors’ values for that feature. The distance is based on all the available features.
2) IterativeImputer trains a regression model per feature to predict the missing values based on all the other available features. It then trains the model again on the updated data, and repeats the process several times, improving the models and the replacement values at each iteration

In [ ]:
Df1 = pd.DataFrame({'NUMERICAL_ATTRIBUTES': NUMERICAL_ATTRIBUTES.columns})
Df1['Feature Median By Imputer'] = imputer.statistics_
Df1['Feature Median By Median Function'] = NUMERICAL_ATTRIBUTES.median().values
Df1


In [ ]:
X_Inputer_train = imputer.transform(NUMERICAL_ATTRIBUTES)

In [ ]:
X_Inputer_train = pd.DataFrame(X_Inputer_train , columns=NUMERICAL_ATTRIBUTES.columns, index=NUMERICAL_ATTRIBUTES.index)
X_Inputer_train.isna().sum().sum()

In [ ]:
X_Inputer_train.info()  # So this the X_Inputer_train contain trainset without any null value cause the Imputer had handle all the null values by the median 

**Handling Text and Categorical Attributes**

Most machine learning algorithms prefer to work with numbers, so let’s convert these categories from text to numbers.

In [ ]:
CATEGORICAL_ATTRIBUTES.head(3)

In [ ]:
CATEGORICAL_ATTRIBUTES['sector'].value_counts()

In [ ]:
CATEGORICAL_ATTRIBUTES['state'].value_counts()

One-hot encoder encode in such a way were one attribute will be equal to 1 (hot), while the others will be 0 (cold). The new attributes are sometimes called dummy attributes. Scikit-Learn provides a OneHotEncoder class to convert categorical values into one-hot vectors

In [ ]:
cat_encoder = OneHotEncoder()
X_encode_train = cat_encoder.fit_transform(CATEGORICAL_ATTRIBUTES)

In [ ]:
cat_encoder.categories_

In [ ]:
X_encode_train = pd.DataFrame(X_encode_train.toarray(), columns=cat_encoder.get_feature_names_out(), index=CATEGORICAL_ATTRIBUTES.index)

In [ ]:
X_encode_train.head(3)

As we can see that, the first sector value is Roads & Highways, So in the One Hot encoded vector value is only hold 1 in the feature sector_Roads & Highways, while all other we hold the value of 0..

**Feature Scaling and Transformation** 

In [ ]:
X_Inputer_train.info()

In [ ]:
X_Inputer_train.describe()

*Machine learning algorithms don’t perform well when the input numerical attributes have very different scales.*

This is the case for the Project data: the Project Cost (Cr) ranges from about 50 cr to 12759.70 cr , while the Contractor Experience Year only range from 1 to 30.
Without any scaling, most models will be biased toward ignoring the Contractor Experience Year and focusing more on the Project Cost (Cr).


There are two common ways to get all attributes to have the same scale:
min-max scaling and standardization.

*Min-max scaling (many people call this normalization) is the simplest: for each attribute, the values are shifted and rescaled so that they end up ranging from 0 to 1.*
*This is performed by subtracting the min value and dividing by the difference between the min and the max.*

In [ ]:
min_max_scalar = MinMaxScaler(feature_range=(-1,1))
X_MinMax_scaled_train = min_max_scalar.fit_transform(X_Inputer_train)

In [ ]:
X_MinMax_scaled_train = pd.DataFrame(X_MinMax_scaled_train, columns=X_Inputer_train.columns, index  = X_Inputer_train.index)
X_MinMax_scaled_train.head(5)

This is How the Min Max Scaler Work..

*Standardization is different: first it subtracts the mean value (so standardized values have a zero mean), then it divides the result by the standard deviation (so standardized* *values have a standard deviation equal to 1)*

In [ ]:
std_scaler = StandardScaler()
X_std_scaled_train = std_scaler.fit_transform(X_Inputer_train)
X_std_scaled_train = pd.DataFrame(X_std_scaled_train, columns=X_Inputer_train.columns, index  = X_Inputer_train.index)
X_std_scaled_train.head(5)

So, which scaler to use:- standardization is much less affected by outliers.

For Example:-  suppose a district has a Contractor Exp year equal to 50 (by mistake), instead of the usual 1–30. 
Min-max scaling to the 0–1 range would map this outlier down to 1 and it would crush all the other values down to 0–0.15, whereas standardization would not be much affected.

In [ ]:
X_std_scaled_train.hist(figsize=(10,10), bins = 30)
plt.show()

When a feature’s distribution has a heavy tail (when values far from the mean are not exponentially rare), both min-max scaling and standardization  will squash most values into a small range.

Machine learning models generally don’t like this at all.
Features like project cost cr is highly right skewed with long tail , such as a power law distribution, then replacing the feature with its logarithm may help and after the we can scale it..

In [ ]:
X_std_scaled_train["log_cost"] = np.log1p(X_std_scaled_train["project_cost_cr"])  

In [ ]:
X_std_scaled_train.hist(figsize=(10,10), bins = 30)
plt.show()

As we can see the Log_cost comes the new feature and this feature after scale will have much more give imp to tth model that the skewwed one..

In [ ]:
# Check approximate means and standard deviations
display(pd.DataFrame({
    "Mean": X_std_scaled_train.mean(),
    "Standard Deviation": X_std_scaled_train.std(ddof=0)
}).round(3))

**Custom Transformers**

In [ ]:
# Custom Transformer can be use tasks such as custom transformations, cleanup operations, or combining specific attributes
# For our need we have to :-  it's often a good idea to transform features with heavy-tailed distributions by replacing them with their logarithm.

# Let Create an Log Transformer - 

log_transformer = FunctionTransformer(np.log1p, inverse_func= np.exp)   # This is our Log Transformer so let use this transformer..

In [ ]:
X_std_scaled_train.head(4)

In [ ]:
X_std_scaled_train['Log_Transformer_Cost'] = log_transformer.transform(X_std_scaled_train['project_cost_cr'])

In [ ]:
raw_cost = X_train["project_cost_cr"]
log_cost = log_transformer.transform(raw_cost)                                   # log of RAW values (always positive)
log_cost_scaled = StandardScaler().fit_transform(log_cost.to_frame())

print(f"skew of raw cost      : {stats.skew(raw_cost):.2f}")
print(f"skew of log(cost)     : {stats.skew(log_cost):.2f}")
print(f"mean/std after scaling: {log_cost_scaled.mean():.2f} / {log_cost_scaled.std():.2f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(raw_cost, bins=30); ax[0].set_title("raw cost")
ax[1].hist(log_cost, bins=30); ax[1].set_title("log(cost)")
plt.tight_layout(); plt.show()

In [ ]:
X_std_scaled_train.head()

**Pipeline - allows you to chain multiple data preprocessing steps and a final estimator into a single, reproducible workflow.**

The preprocessing workflow uses three transformations:

Numerical features: median imputation followed by standardization.

Categorical features: most-frequent imputation followed by one-hot encoding.

Project cost: median imputation, logarithmic transformation, and standardization.

Project cost is handled separately because its right-skewed distribution may benefit from a logarithmic transformation. The transformation is applied to the original cost values before scaling.

*Preprocessing Pipeline For the Numerical Attributes*

*Preprocessing Pipeline For the Categorical Attributes*

In [ ]:

# 1. General numerical features
num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

# 2. Categorical features
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

# 3. Project cost: impute -> log transform -> scale
log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(                           # Function Transformer
        np.log1p,
        feature_names_out="one-to-one"
    ),
    StandardScaler()
)


In [ ]:
NUMERICAL_ATTRIBUTES_COL = NUMERICAL_ATTRIBUTES.columns
CATEGORICAL_ATTRIBUTES_COL = CATEGORICAL_ATTRIBUTES.columns

**ColumnTransformer - allows you to apply different preprocessing transformations to specific columns of a dataset**

In [ ]:
preprocessing = ColumnTransformer([
    ('num', num_pipeline,NUMERICAL_ATTRIBUTES_COL),
    ('cat', cat_pipeline ,CATEGORICAL_ATTRIBUTES_COL)
])

In [ ]:
x_train_copy_prepared = preprocessing.fit_transform(x_train_copy)

In [ ]:
x_train_copy_prepared = pd.DataFrame(x_train_copy_prepared, columns=preprocessing.get_feature_names_out(), index= x_train_copy.index)
print(x_train_copy_prepared.shape)
x_train_copy_prepared.head(3)

**Feature engineering - user-defined class that allows you to implement unique data preprocessing steps and integrate them seamlessly into pipelines.**

About our Project we have 4 feature that has significant corelation with the Target column with will help model to gain more understanding

In [ ]:

# class LandRiskFeatureEngineer(BaseEstimator, TransformerMixin):
#     """Create project-status features used for delay-risk prediction."""

#     def fit(self, X, y=None):
#         return self

#     def transform(self, X):
#         X = X.copy()

#         duration = X["planned_duration_months"]

#         # Avoid division by zero or invalid planned durations.
#         if (duration <= 0).any():
#             raise ValueError(
#                 "Planned duration must be greater than zero."
#             )

#         # Percentage of planned time already consumed.
#         X["time_elapsed_pct"] = (
#             X["months_elapsed"] / duration
#         ) * 100

#         # Negative values indicate progress behind the
#         # simple time-proportional benchmark.
#         X["schedule_slippage"] = (
#             X["physical_progress_pct"]
#             - X["time_elapsed_pct"]
#         )

#         # Positive values indicate expenditure ahead of
#         # physical progress.
#         X["spending_progress_gap"] = (
#             X["expenditure_pct"]
#             - X["physical_progress_pct"]
#         )

#         # Remaining acquisition percentage.
#         X["land_gap"] = (
#             100 - X["land_acquisition_pct"]
#         )

#         # Land acquisition is irrelevant when land is not required.
#         no_land_required = (
#             X["land_acquisition_required"] == 0
#         )
#         X.loc[no_land_required, "land_gap"] = 0

#         return X

#     def get_feature_names_out(self, input_features=None):
#         if input_features is None:
#             input_features = [
#                     "sector",
#                     "state",
#                     "project_cost_cr",
#                     "planned_duration_months",
#                     "months_elapsed",
#                     "physical_progress_pct",
#                     "expenditure_pct",
#                     "contractor_experience_years",
#                     "land_acquisition_required",
#                     "land_acquisition_pct"
#                 ]

#         return np.asarray(
#             list(input_features) + [
#                     "time_elapsed_pct",
#                     "schedule_slippage",
#                     "spending_progress_gap",
#                     "land_gap",
#                 ],
#             dtype=object,
#             )

In [ ]:
print(X_train.columns.tolist())

In [ ]:
# Over All preprocessing - 


preprocessing = ColumnTransformer([
    ("log", log_pipeline, ['project_cost_cr']),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),    
],
remainder = num_pipeline
)

In [ ]:
main_preprocessing_pipeline = make_pipeline(
    preprocessing
)

In [ ]:
X_Preprocessed_train = main_preprocessing_pipeline.fit_transform(X_train)
X_Preprocessed_train = pd.DataFrame(X_Preprocessed_train, columns=main_preprocessing_pipeline.get_feature_names_out(), index = X_train.index)

In [ ]:
X_Preprocessed_train.head()

In [ ]:
X_Preprocessed_train.info()

In [ ]:
print("Original training shape:", X_train.shape)
print("Prepared training shape:", X_Preprocessed_train.shape)

**Model Building** 

Now as we can see that the, data set is currently well preprocessing, feature engineered..So now let see how the model is build and trained..

Evaluation Strategy

The dataset is divided into training and test sets. Model development uses the training set, while the test set is reserved for final evaluation.

To compare classifiers consistently, we use stratified three-fold cross-validation on the training data. Each fold preserves approximately the same class distribution.

The complete machine-learning pipeline—including feature engineering, imputation, encoding, scaling, and classification—is fitted separately within each training fold. This prevents learned preprocessing steps from using validation-fold information.

The main metrics are accuracy, precision, recall, F1-score, and ROC-AUC. Because LandRisk AI is an early-warning prototype, recall for the high-risk class is especially important, but it must be considered alongside precision.

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [ ]:
y_train.value_counts(normalize= True)

In [ ]:
y_train.head()

In [ ]:
# SGD  good place to start is with a stochastic gradient descent (SGD, or stochastic GD) classifier, using Scikit Learn’s SGDClassifier class

sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_Preprocessed_train, y_train)


In [ ]:
# Now let see does this model work's or not?

y_train.iloc[500:506]

In [ ]:
sgd_clf.predict(X_Preprocessed_train.iloc[500:506])

*Here the Prediction is All Right in this particular case! Now, let’s evaluate this model’s performance* 

Performance Measures - 

Measuring Accuracy Using Cross-Validation

In [ ]:
cross_val_score(sgd_clf, 
                X_Preprocessed_train, 
                y_train, 
                cv= cv, 
                scoring="accuracy")

Here we can see that we got around 70-72 pct accuracy..an all cross-validation folds..but let build and dummy classifiers this dummy classifires will predict only one part of the prediction which is 0 project will have low Delay Risk..

Establish a baseline
A professional model comparison should include a simple reference point. 
Your earlier DummyClassifier experiment is useful, but it used the already-preprocessed training matrix.


In [ ]:
baseline_pipeline = make_pipeline(
    clone(LandRiskFeatureEngineer()),
    clone(main_preprocessing_pipeline),
    DummyClassifier(strategy="most_frequent")
)

baseline_predictions = cross_val_predict(
    baseline_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict"
)

print("Baseline accuracy:",
      round(accuracy_score(y_train, baseline_predictions), 3))

print("\nBaseline classification report:")
print(classification_report(
    y_train,
    baseline_predictions,
    zero_division=0
))

In [ ]:
# Here we can see that the dummy-classifier whose just simply predicting 0 (project have low delay risk) for every project get the accuracy around 63-64 pct..

# This demonstrates why accuracy is generally not the preferred performance measure for classifiers, especially when you are dealing with skewed datasets (i.e., when some classes are much more frequent than others). 
# A much better way to evaluate the performance of a classifier is to look at the confusion matrix (CM).


In [ ]:
# Confusion Matrices - The general idea of a confusion matrix is to count the number of times instances of class A are classified as class B, for all A/B pairs
# For example - to know the number of times the classifier confused to classify of 0 (project have low delay risk) with 1 (project have high delay risk), you would look at row #0, column #1 of the confusion matrix.



In [ ]:
y_train_pred = cross_val_predict(sgd_clf,          # cross_val_predict will predict and returns the predictions made on each test fold.
                                X_Preprocessed_train, 
                                y_train,   
                                cv= cv)

In [ ]:
cm = confusion_matrix(y_train, y_train_pred)
cm

In [ ]:
print(873 + 151)   # So in the confusions matrix 1st row represents projects with low delay risk (0).
print(302 + 274)   # and the 2nd row represents projects with high delay risk (1).

Each row in a confusion matrix represents an actual class, while each column represents a predicted class.

The first row of this matrix considers projects with the low delay risk [0] (the negative class): 833 of them were correctly classified as low delay risk (they are called true negatives).
while the remaining 191 were wrongly classified as the high delay risk [1](false positives, also called type I errors)

The second row considers the Project with high delay risk [1](the positive class): 324 were wrongly classified as Project with low delay risk [0] (false negatives, also called type II errors)
while the remaining 252 were correctly classified as project with high delay risk [1] (true positives).

*So prefect classifier would only have true positiive and true negative*



This gives us an idea about how the model predictions was actually were...but now lets see the precision..
precision = TP / (TP + FP)

In [ ]:
precision_score(y_train, y_train_pred)

precision is typically used along with another metric named recall, also called sensitivity or the true positive rate (TPR) 
recall = TP /  (TP +FN)

In [ ]:
recall_score(y_train, y_train_pred)

So , by this we can say that the model (SGD Classifier) only help us by 64.45 % all the time.
while it detect only about 47.5% project as high delay risk project.

In [ ]:
f1_score(y_train, y_train_pred)

Now let work with the Precision and recall...

Which is more important for LandRisk AI? 
Precision or Recall 

If Precision as Imp :- 
1) False positive: You flag a project as high risk when it isn't. People might waste time investigating it or allocate unnecessary resources.
2) False negative: You label a genuinely high-risk project as low risk. People might fail to intervene early, potentially allowing a delay to become worse.

For an early-warning system like LandRisk AI, recall may be at least as important as precision. 

In [ ]:
y_scores = cross_val_predict(sgd_clf,
                            X_Preprocessed_train, 
                            y_train,   
                            cv= 3,
                            method = "decision_function"
)

In [ ]:
# Here, we got the decisions scores of the prediction that model use to make the prediction.
# An SGD classifier produces a decision score. The threshold determines how high that score must be for a project to be classified as high risk.

# Think of the threshold as the strictness of your warning system.

# High threshold — stricter warnings
# The model flags fewer projects as high risk. This often increases precision but lowers recall because more genuinely high-risk projects may be missed.

# Low threshold — more warnings
# The model flags more projects as high risk. This often increases recall but lowers precision because more low-risk projects may be incorrectly flagged.

precisions, recalls, thresholds = precision_recall_curve(y_train, y_scores)


In [ ]:

plt.figure(figsize=(10, 6))

# Plot precision and recall against the decision threshold
plt.plot(
    thresholds,
    precisions[:-1],
    "b-",
    linewidth=2,
    label="Precision"
)

plt.plot(
    thresholds,
    recalls[:-1],
    "g-",
    linewidth=2,
    label="Recall"
)

# Mark the chosen threshold
threshold = 1.2

plt.axvline(
    x=threshold,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Threshold = {threshold}"
)

# Improve graph appearance
plt.xlabel("Decision Threshold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.title("Precision and Recall vs Decision Threshold", fontsize=14)

plt.ylim(0, 1.05)
plt.grid(True, linestyle=":", alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()


In [ ]:

plt.figure(figsize=(8, 6))

plt.plot(
    recalls[:-1],
    precisions[:-1],
    linewidth=2,
    color="purple"
)

plt.xlabel("Recall", fontsize=12)
plt.ylabel("Precision", fontsize=12)
plt.title("Precision–Recall Curve: SGD Classifier", fontsize=14)

plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()

plt.show()


For LandRisk AI:
- Prioritize recall so we miss fewer genuinely high-risk projects.
- Require precision to remain at a reasonable level, so the system doesn't flag too many low-risk projects.
- Compare several thresholds using cross-validated predictions, then choose one based on that trade-off.

In [ ]:

# Calculate precision and recall for possible thresholds
precisions, recalls, thresholds = precision_recall_curve(
    y_train,
    y_scores
)

In [ ]:
# Among all thresholds with precision of at least 60%, choose the threshold that achieves the highest recall.

minimum_precision = 0.60

# Precision and recall correspond to thresholds[:-1]
valid_indices = np.where(precisions[:-1] >= minimum_precision)[0]

# print(np.where(precisions[:-1] >= minimum_precision)[0])

if len(valid_indices) == 0:
    print("No threshold satisfies the minimum precision.")

else:
    # Find the valid threshold with the highest recall
    best_index = valid_indices[np.argmax(recalls[:-1][valid_indices])]

    best_threshold = thresholds[best_index]

    print("Chosen threshold:", best_threshold)
    print("Precision:", precisions[best_index])
    print("Recall:", recalls[best_index])


In [ ]:

plt.figure(figsize=(10, 6))

# Plot precision and recall against the decision threshold
plt.plot(
    thresholds,
    precisions[:-1],
    "b-",
    linewidth=2,
    label="Precision"
)

plt.plot(
    thresholds,
    recalls[:-1],
    "g-",
    linewidth=2,
    label="Recall"
)

# Mark the chosen threshold
threshold = 1.2

plt.axvline(
    x=best_threshold,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Threshold = {best_threshold}"
)

# Improve graph appearance
plt.xlabel("Decision Threshold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.title("Precision and Recall vs Decision Threshold", fontsize=14)

plt.ylim(0, 1.05)
plt.grid(True, linestyle=":", alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()


In [ ]:
y_train_pred_threshold = (y_scores >= best_threshold).astype(int)

print(confusion_matrix(y_train, y_train_pred_threshold))

In [ ]:
print(classification_report(
    y_train,
    y_train_pred_threshold,
    zero_division=0
))

In [ ]:
# Here we can see that the precision score for predicting 1 is around 60% and recall is around 58%.

**ROC Curve**  

The receiver operating characteristic (ROC) curve is another common tool used with binary classifiers. 
- ROC curve plots the true positive rate (another name for recall) against the false positive rate (FPR).
- The FPR (also called the fall-out) is the ratio of negative instances that are incorrectly classified as positive. 
- It is equal to 1 – the true negative rate (TNR), which is the ratio of negative instances that are correctly classified as negative. 
- The TNR is also called specificity
- Hence, the ROC curve plots sensitivity (recall) versus 1 – specificity

In [ ]:
# Calculate ROC points from out-of-fold scores
fpr, tpr, roc_thresholds = roc_curve(
    y_train,
    y_scores
)

roc_auc = roc_auc_score(y_train, y_scores)

# Find the ROC point nearest to our selected threshold
idx = np.argmin(
    np.abs(roc_thresholds - best_threshold)
)

fpr_selected = fpr[idx]
tpr_selected = tpr[idx]

# Plot
plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"SGD Classifier (AUC = {roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    "k:",
    label="Random classifier"
)

plt.scatter(
    fpr_selected,
    tpr_selected,
    color="red",
    s=80,
    zorder=3,
    label=f"Selected threshold = {best_threshold:.2f}"
)

plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate (Recall)", fontsize=12)
plt.title("ROC Curve — LandRisk AI", fontsize=14)

plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.grid(True, linestyle=":", alpha=0.5)
plt.legend(loc="lower right")
plt.tight_layout()

plt.show()

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"FPR at selected threshold: {fpr_selected:.4f}")
print(f"TPR at selected threshold: {tpr_selected:.4f}")

The dotted line represents the ROC curve of a purely random classifier; a good classifier stays as far away from that line as possible (toward the top-left corner).

One way to compare classifiers is to measure the area under the curve (AUC). 
A perfect classifier will have a ROC AUC equal to 1, whereas a purely random classifier will have a ROC AUC equal to 0.5.

An ROC-AUC of approximately 0.741 means the model has a reasonable ability to rank high-risk projects above low-risk projects.

In [ ]:
roc_auc_score(y_train , y_scores)

Now Let See some other classification models to improve our work..
Compare the candidate classifiers
Keep Logistic Regression, Decision Tree, and Random Forest as main candidates.

### 4.4 Model Comparison — Initial Findings

The models are compared using the same stratified cross-validation procedure.

Logistic Regression is the leading initial candidate based on the previously observed ROC-AUC and F1-score. Its relatively simple structure also makes it a useful starting point for interpreting which features influence predictions.

Random Forest is a competitive alternative, but its performance must be evaluated alongside model complexity and the precision–recall trade-off.

The Decision Tree provides a simpler nonlinear benchmark, although its earlier results were weaker.

These findings are preliminary. The model selected for further development will be tuned using training data, and its final performance will be measured on the untouched test set.

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}


pipelines = {
    name: make_pipeline(
        clone(LandRiskFeatureEngineer()),
        clone(main_preprocessing_pipeline),
        clone(model)
    )
    for name, model in models.items()
}

results = []

for name, pipeline in pipelines.items():

    # Obtain out-of-fold probabilities for high risk (class 1).
    probabilities = cross_val_predict(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    # Use the default 0.50 threshold for initial comparison.
    predictions = (probabilities >= 0.50).astype(int)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_train, predictions),
        "Precision": precision_score(
            y_train, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_train, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_train, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_train, probabilities)
    })

results_df = (
    pd.DataFrame(results)
    .set_index("Model")
    .sort_values("ROC-AUC", ascending=False)
)

display(results_df.round(3))



**Logistic Regression — current leading baseline**

ROC-AUC 0.77,
Recall 0.53

It has the strongest ROC-AUC, highest F1-score (0.58), and ties for the best high-risk recall. It's our first candidate for tuning.


**Decision Tree — weaker baseline**

ROC-AUC 0.61,
Recall 0.53

It identifies a similar proportion of high-risk projects at the default threshold, but its ranking ability and precision are weaker. We can test whether controlling tree complexity helps.


**Random Forest — competitive alternative**

ROC-AUC 0.76,
Recall 0.48

It performs nearly as well as Logistic Regression on ROC-AUC but has lower high-risk recall at the current default threshold.

The Decision Tree model maybe not good to train..

So, the Random Forest Model work with the rules of the  wisdom of the crowd, which simply mean that 
if you aggregate the predictions of a group of predictors (such as classifiers or regressors), you will often get better predictions than with the best individual predictor. 
A group of predictors is called an ensemble; thus, this technique is called ensemble learning.

As an example of an ensemble method, you can train a group of decision tree classifiers, each on a different random subset of the training set. 
You can then obtain the predictions of all the individual trees, and the class that gets the most votes is the ensemble’s prediction.
Such an ensemble of decision trees is called a random forest

**Random Forest** - a random forest is an ensemble of decision trees, generally trained via the bagging method (or sometimes pasting), typically with max_samples set to the size of the training set. 

But Bagging Method ?
When sampling (Getting some part of the Whole Dataset to train or inspect) is performed with replacement, this method is called bagging.




Voting Classifier - A very simple way to create an even better classifier is to aggregate the predictions of each classifier: the class that gets the most votes is the ensemble’s prediction. 
This majority-vote classifier is called a hard voting classifier.

In [ ]:
Voting_clf = VotingClassifier(
    estimators = [
        ("lr", LogisticRegression(random_state=42)),
        ("dt", DecisionTreeClassifier(random_state=42)),
        ("svc",SVC(random_state=42)),
        ("sgd", SGDClassifier(random_state=42))
    ]
)

Voting_clf_accuracy = cross_val_score(
    Voting_clf,
    X_Preprocessed_train,
    y_train,
    cv=3,
    scoring="accuracy"
)


print("Accuracy scores:", Voting_clf_accuracy)
print("Mean CV accuracy:", Voting_clf_accuracy.mean())


In [ ]:
Voting_clf.fit(X_Preprocessed_train, y_train)

In [ ]:
Voting_clf.predict(X_Preprocessed_train.iloc[500:506])

In [ ]:
y_train[500:506]

Here the Predict is too not great even tho it have an Group of multiple classifier..

In [ ]:
voting_clf_predictions = cross_val_predict(
    Voting_clf,
    X_Preprocessed_train,
    y_train,
    cv=3
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_train, voting_clf_predictions))

print("\nClassification Report:")
print(classification_report(
    y_train,
    voting_clf_predictions,
    zero_division=0
))

In [ ]:
# 3. Out-of-fold probability scores for ROC-AUC
voting_clf_scores = cross_val_predict(
    Voting_clf,
    X_Preprocessed_train,
    y_train,
    cv=3,
)

print(
    "\nVoting Classifier ROC-AUC:",
    roc_auc_score(y_train, voting_clf_scores)
)

Here the Voting Classifier Didn't heklp us muchh

***Hyperparameter tuning***  -- 

Logistic Regression is our leading baseline. Its C parameter controls regularization:
- Smaller C → stronger regularization, which restricts model complexity.
- Larger C → weaker regularization, allowing the model to fit the training data more closely.

In [ ]:
lr_param_grid = {
    "logisticregression__C": [0.01, 0.1, 1, 10, 100]
}

lr_grid_search = GridSearchCV(
    estimator=pipelines["Logistic Regression"],
    param_grid=lr_param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

lr_grid_search.fit(X_train, y_train)

print("Best parameters:", lr_grid_search.best_params_)
print("Best mean CV ROC-AUC:",
      round(lr_grid_search.best_score_, 4))

In [ ]:
rf_param_grid = {
    "randomforestclassifier__max_depth": [5, 10, 20, None],
    "randomforestclassifier__min_samples_leaf": [1, 5, 10],
    "randomforestclassifier__max_features": ["sqrt", 0.7]
}

rf_grid_search = GridSearchCV(
    estimator=pipelines["Random Forest"],
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

rf_grid_search.fit(X_train, y_train)

print("Best parameters:", rf_grid_search.best_params_)
print("Best mean CV ROC-AUC:",
      round(rf_grid_search.best_score_, 4))

In [ ]:
rf_results = pd.DataFrame(rf_grid_search.cv_results_)

print(
    rf_results[
        [
            "param_randomforestclassifier__max_depth",
            "param_randomforestclassifier__min_samples_leaf",
            "param_randomforestclassifier__max_features",
            "mean_test_score",
            "std_test_score",
            "mean_train_score"
        ]
    ]
    .sort_values("mean_test_score", ascending=False)
    .head(5)
    .round(3)
)

In [ ]:
# Compare the tuned models
# Model	                          |     Best mean CV ROC-AUC
# _________________________________________________________________
# Logistic Regression	          |         0.7775
# Random Forest	                  |         0.7762

In [ ]:
best_lr_model = lr_grid_search.best_estimator_

lr_oof_probabilities = cross_val_predict(
    best_lr_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

The difference is only about 0.0013. That's negligible for practical model selection at this stage.
We shouldn't declare a definitive winner based on that difference. Logistic Regression is simpler and easier to interpret, so it remains a strong candidate. Random Forest remains worth considering if it offers a better recall/precision trade-off.

In [ ]:
# Select the best Logistic Regression pipeline
best_lr_model = lr_grid_search.best_estimator_

# Candidate threshold chosen from the earlier exploratory analysis
final_threshold = 0.35

print("Best parameters:", lr_grid_search.best_params_)
print("Best cross-validation ROC-AUC:", lr_grid_search.best_score_)
print("Selected classification threshold:", final_threshold)

In [ ]:

# Predict probabilities using raw test features
test_probabilities = best_lr_model.predict_proba(X_test)[:, 1]

# Apply the selected threshold
test_predictions = (
    test_probabilities >= final_threshold
).astype(int)

print("Accuracy:", accuracy_score(y_test, test_predictions))
print("High-risk precision:", precision_score(y_test, test_predictions, zero_division=0))
print("High-risk recall:", recall_score(y_test, test_predictions, zero_division=0))
print("High-risk F1:", f1_score(y_test, test_predictions, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, test_probabilities))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=["Lower Risk", "Higher Risk"],
    zero_division=0
))

Our project's purpose is early detection of potential delays. Missing a genuinely high-risk project could be more costly than asking someone to review a project that turns out to be fine.

We need to define what good means for this project. Here are three reasonable strategies:

A. Maximize F1-score
Balance precision and recall. A reasonable starting point when the cost of false alarms versus missed risks is not yet known.

B. Prioritize recall
Choose the threshold with the highest precision while satisfying a recall requirement—for example, recall of at least 75%, if that target is justified.

C. Minimize business cost
Assign costs to missed high-risk projects and false alarms, then choose the threshold with the lowest expected cost. This is best when defensible cost estimates exist.

Let Go with the Strategy B as the project design goal, with a recall target chosen and documented before using the test set to evaluate it.

For LandRisk AI threshold selection, prioritize recall with false-alarm monitoring. Minimum recall target: 75%. Help implement leakage-safe threshold selection before test evaluation.

In [ ]:
from sklearn.model_selection import StratifiedKFold


# Outer CV: creates predictions for held-out folds
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Inner CV: tunes Logistic Regression within each outer training fold
inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=123
)

param_grid = {
    "logisticregression__C": [0.01, 0.1, 1, 10, 100]
}

# This is the complete pipeline: feature engineering + preprocessing + model
base_pipeline = clone(pipelines["Logistic Regression"])

# Store one out-of-fold probability for every training observation
oof_probabilities = np.zeros(len(X_train), dtype=float)

for fold_number, (train_idx, valid_idx) in enumerate(
    outer_cv.split(X_train, y_train),
    start=1
):
    X_outer_train = X_train.iloc[train_idx]
    y_outer_train = y_train.iloc[train_idx]

    X_outer_valid = X_train.iloc[valid_idx]

    # Tune hyperparameters using only this outer training fold
    inner_search = GridSearchCV(
        estimator=clone(base_pipeline),
        param_grid=param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1
    )

    inner_search.fit(X_outer_train, y_outer_train)

    # Predict probabilities for observations excluded from model fitting
    oof_probabilities[valid_idx] = (
        inner_search.best_estimator_
        .predict_proba(X_outer_valid)[:, 1]
    )

    print(
        f"Outer fold {fold_number}: "
        f"best C = {inner_search.best_params_['logisticregression__C']}"
    )

print("OOF probabilities generated:", len(oof_probabilities))
print("Training observations:", len(X_train))

In [ ]:
threshold_rows = []

for threshold in np.arange(0.05, 0.96, 0.01):
    predictions = (oof_probabilities >= threshold).astype(int)

    precision = precision_score(
        y_train, predictions, zero_division=0
    )
    recall = recall_score(
        y_train, predictions, zero_division=0
    )
    f1 = f1_score(
        y_train, predictions, zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_train, predictions, labels=[0, 1]
    ).ravel()

    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "true_negatives": tn
    })

threshold_results = pd.DataFrame(threshold_rows)

# Enforce the minimum recall requirement
eligible_thresholds = threshold_results[
    threshold_results["recall"] >= 0.75
]

if eligible_thresholds.empty:
    print("No tested threshold achieved 75% recall.")
    print("Review the recall/precision trade-off before proceeding.")
else:
    # Maximize precision; if tied, prefer the higher threshold
    selected_row = eligible_thresholds.sort_values(
        ["precision", "threshold"],
        ascending=[False, False]
    ).iloc[0]

    selected_threshold = float(selected_row["threshold"])

    print("Selected threshold:", selected_threshold)
    print("OOF precision:", round(selected_row["precision"], 3))
    print("OOF recall:", round(selected_row["recall"], 3))
    print("OOF F1-score:", round(selected_row["f1"], 3))
    print("False positives:", int(selected_row["false_positives"]))
    print("False negatives:", int(selected_row["false_negatives"]))

    display(
        eligible_thresholds.sort_values(
            "threshold"
        ).round(3)
    )

In [ ]:
final_search = GridSearchCV(
    estimator=clone(pipelines["Logistic Regression"]),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=inner_cv,
    n_jobs=-1,
    refit=True
)

final_search.fit(X_train, y_train)

final_model = final_search.best_estimator_

print("Final model parameters:", final_search.best_params_)
print("Locked threshold:", selected_threshold)

In [ ]:
test_probabilities = final_model.predict_proba(X_test)[:, 1]
test_predictions = (
    test_probabilities >= selected_threshold
).astype(int)

print("Threshold:", selected_threshold)
print("Accuracy:", accuracy_score(y_test, test_predictions))
print("High-risk precision:", precision_score(
    y_test, test_predictions, zero_division=0
))
print("High-risk recall:", recall_score(
    y_test, test_predictions, zero_division=0
))
print("High-risk F1:", f1_score(
    y_test, test_predictions, zero_division=0
))
print("ROC-AUC:", roc_auc_score(y_test, test_probabilities))
print("Confusion matrix:\n", confusion_matrix(y_test, test_predictions))
print(classification_report(
    y_test,
    test_predictions,
    target_names=["Lower Risk", "Higher Risk"],
    zero_division=0
))

In [ ]:
final_model

In [ ]:
import json
import hashlib
import platform
from datetime import datetime, timezone

import joblib
import sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# ---- 1. Fixed paths, built from ROOT (the repo root), not from where the notebook runs
ARTIFACT_DIR = ROOT / "ml" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATH = ARTIFACT_DIR / "landrisk_model.joblib"      # model + metadata (pickle, for Django)
MODEL_CARD_PATH = ARTIFACT_DIR / "model_card.json"          # same metadata, readable by humans
DATA_FILE = ROOT / "ml" / "data" / "infrastructure_projects.csv"

# ---- 2. Input schema: what the app must collect, and what is allowed.
# YOU decide the numbers marked EDIT. Keep them consistent with docs/DATA_CARD.md.
NUMERIC_RULES = {
    "project_cost_cr":             {"type": "float", "required": True,  "min": 1, "max": 100000},   # EDIT
    "planned_duration_months":     {"type": "int",   "required": True,  "min": 1, "max": 120},      # EDIT
    "months_elapsed":              {"type": "int",   "required": True,  "min": 1, "max": 120},      # EDIT
    "physical_progress_pct":       {"type": "float", "required": True,  "min": 0, "max": 100},
    "expenditure_pct":             {"type": "float", "required": False, "min": 0, "max": 100},      # may be missing
    "contractor_experience_years": {"type": "int",   "required": False, "min": 0, "max": 60},       # EDIT, may be missing
    "land_acquisition_required":   {"type": "int",   "required": True,  "allowed_values": [0, 1]},
    "land_acquisition_pct":        {"type": "float", "required": True,  "min": 0, "max": 100},
}

# Rules that involve more than one field. inference.py will enforce them.
CROSS_FIELD_RULES = [
    "months_elapsed must not be greater than planned_duration_months (the model was trained only on projects before their deadline)",
    "if land_acquisition_required == 0 then land_acquisition_pct must be 100",
]

input_schema = {}
for col in ["sector", "state"]:                              # categories come from the training data
    input_schema[col] = {
        "type": "category",
        "required": True,
        "allowed_values": sorted(X_train[col].dropna().unique().tolist()),
    }
for col, rule in NUMERIC_RULES.items():
    input_schema[col] = dict(rule)
    # Also record what the training data looked like (useful to warn about unusual inputs)
    input_schema[col]["training_range"] = [float(X_train[col].min()), float(X_train[col].max())]

# Safety check: the schema must describe exactly the columns the model was trained on
assert set(input_schema) == set(X_train.columns), (
    f"schema and training columns differ: {set(input_schema) ^ set(X_train.columns)}"
)

# ---- 3. Metrics: read from the variables, never typed by hand
# (the SAME numbers you already reported; nothing new is being decided here)
test_probabilities = final_model.predict_proba(X_test)[:, 1]
oof_pred = (oof_probabilities >= selected_threshold).astype(int)
test_pred = (test_probabilities >= selected_threshold).astype(int)

def summarize(y_true, probabilities, predictions):
    return {
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
    }

metrics = {
    "out_of_fold_train": summarize(y_train, oof_probabilities, oof_pred),
    "test": summarize(y_test, test_probabilities, test_pred),
}

# ---- 4. Training info and library versions
training_info = {
    "train_rows": int(len(X_train)),
    "delayed_share_train": float(y_train.mean()),
    "random_state": 42,
    "final_C": float(final_model[-1].C),              # last pipeline step = LogisticRegression
    "data_file": DATA_FILE.name,
    "data_sha256": hashlib.sha256(DATA_FILE.read_bytes()).hexdigest(),   # fingerprint of the exact data used
}
library_versions = {
    "python": platform.python_version(),
    "scikit-learn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "joblib": joblib.__version__,
}

# ---- 5. Everything in one dictionary
artifact = {
    "model": final_model,
    "model_name": "Logistic Regression",
    "model_version": "1.0.0",
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "threshold": float(selected_threshold),
    "target_recall": 0.75,
    "threshold_rule": "Highest threshold whose out-of-fold recall on the training set is at least 0.75",
    "input_schema": input_schema,
    "cross_field_rules": CROSS_FIELD_RULES,
    "training_info": training_info,
    "library_versions": library_versions,
    "metrics": metrics,
    "limitations": [
        "Trained on synthetic data: results show the pipeline works, not real-world accuracy.",
        "The test set was evaluated twice (first with C=0.01 and threshold 0.35, then with the final settings).",
        "C and the threshold were chosen with cross-validation on the same training data, so training scores are slightly optimistic.",
        "Probability calibration has not been verified yet; show risk bands, not exact percentages, until it is.",
    ],
}

joblib.dump(artifact, ARTIFACT_PATH)

# Human-readable copy WITHOUT the model object. json.dumps fails loudly if a NumPy type slipped in.
model_card = {key: value for key, value in artifact.items() if key != "model"}
MODEL_CARD_PATH.write_text(json.dumps(model_card, indent=2), encoding="utf-8")

# ---- 6. Self-check: reload and compare
reloaded = joblib.load(ARTIFACT_PATH)
sample = X_test.iloc[[0]]
print("Saved:", ARTIFACT_PATH)
print("Keys:", list(reloaded.keys()))
print("Same probability after reload:",
      np.isclose(final_model.predict_proba(sample)[0, 1], reloaded["model"].predict_proba(sample)[0, 1]))
print(json.dumps(reloaded["metrics"], indent=2))

In [ ]:
type(final_model.steps[0][1]).__module__

In [ ]:
X_test.head()

In [ ]:
X_test.iloc[5]

In [ ]:
y_test.iloc[[5]]

In [ ]:
print(final_model)
print("\nPipeline steps:")

for name, step in final_model.steps:
    print(f"{name}: {type(step).__name__}")

print("\nFinal estimator:")
print(final_model[-1])

In [ ]:
for name, step in final_model.steps:
    if hasattr(step, "steps"):
        print(f"\nNested steps inside '{name}':")
        for nested_name, nested_step in step.steps:
            print(f"  {nested_name}: {type(nested_step).__name__}")

In [ ]:
import numpy as np

# Select one project from the test set
sample = X_test.iloc[[0]].copy()

# Step 1: Apply feature engineering
engineered = final_model.named_steps[
    "landriskfeatureengineer"
].transform(sample)

# Step 2: Apply fitted preprocessing
preprocessor = final_model.named_steps[
    "pipeline"
].named_steps["columntransformer"]

transformed = preprocessor.transform(engineered)

# Step 3: Get transformed feature names and model coefficients
feature_names = preprocessor.get_feature_names_out()
coefficients = final_model.named_steps[
    "logisticregression"
].coef_[0]

print("Transformed shape:", transformed.shape)
print("Number of feature names:", len(feature_names))
print("Number of coefficients:", len(coefficients))

print("\nFirst 15 transformed features:")
for name, coefficient in zip(
    feature_names[:25], coefficients[:25]
):
    print(f"{name}: {coefficient:.4f}")

In [ ]:
# Convert sparse output to a dense array if necessary
if hasattr(transformed, "toarray"):
    transformed_array = transformed.toarray()
else:
    transformed_array = np.asarray(transformed)

# Each feature's contribution to the logit
contributions = coefficients * transformed_array[0]

# Reconstruct the logit
model = final_model.named_steps["logisticregression"]
reconstructed_logit = model.intercept_[0] + contributions.sum()

# Convert reconstructed logit into probability
reconstructed_probability = 1 / (
    1 + np.exp(-reconstructed_logit)
)

# Compare against the complete pipeline's probability
actual_probability = final_model.predict_proba(sample)[0, 1]

print("Reconstructed probability:", reconstructed_probability)
print("Actual model probability:", actual_probability)
print(
    "Explanation verified:",
    np.isclose(
        reconstructed_probability,
        actual_probability,
        atol=1e-8
    )
)

In [ ]:

import importlib
import ml.explain

importlib.reload(ml.explain)

result = ml.explain.explain_prediction(
    final_model,
    X_test.iloc[[0]].copy(),
)

print("Verified:", result["verified"])
print("Drivers increasing risk:", result["drivers_up"])
print("Drivers decreasing risk:", result["drivers_down"])
